# Linear algebra lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/linear-algebra.ipynb)

The four pieces of linear algebra that LLMs lean on most: dot products, matrix multiplication, low-rank structure, and the LoRA trick built on it. Companion to [Linear algebra](../prereqs/linear-algebra.md). Runs on CPU in seconds.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import numpy as np
import matplotlib.pyplot as plt
from s2a.linalg import cosine_similarity, low_rank_approx, relative_error, lora_params

rng = np.random.default_rng(0)

## 1. Dot products measure agreement

Multiply matching entries and add them up. Big and positive means the vectors point the same way. Cosine similarity is the dot product with the lengths divided out.

In [ ]:
# Made-up 3-number "embeddings": [is an animal, is a pet, is a vehicle]
words = {"cat": [0.9, 0.8, 0.0], "dog": [0.9, 0.9, 0.1], "wolf": [0.9, 0.1, 0.0], "car": [0.0, 0.1, 0.9]}
for a, b in [("cat", "dog"), ("cat", "wolf"), ("cat", "car")]:
    va, vb = np.array(words[a]), np.array(words[b])
    print(f"{a:>4} vs {b:<4}: dot {va @ vb:.2f}, cosine {cosine_similarity(va, vb):.2f}")

Attention scores are exactly this: a dot product between a query and a key.

## 2. A matrix is a transformation

Multiplying points by a matrix moves them all in a consistent way: rotate, stretch, squash. Doing one after another is the same as multiplying the matrices.

In [ ]:
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=float)
t = np.pi / 6
rotate = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
stretch = np.array([[2.0, 0.0], [0.0, 0.5]])

for name, M in [("original", np.eye(2)), ("rotate", rotate), ("stretch", stretch), ("stretch then rotate", rotate @ stretch)]:
    pts = square @ M.T                  # apply M to every point (points are rows)
    plt.plot(pts[:, 0], pts[:, 1], label=name)
plt.axis("equal"); plt.legend(); plt.show()

p = np.array([1.0, 1.0])
print("one step at a time:", rotate @ (stretch @ p), "| combined matrix:", (rotate @ stretch) @ p)
print("order matters:", (stretch @ rotate) @ p)

## 3. Shapes: the rule that prevents most bugs

`(n, d) @ (d, k)` gives `(n, k)`. The inner sizes must match.

In [ ]:
tokens = rng.normal(size=(5, 16))        # 5 tokens, each a 16-number vector
W = rng.normal(size=(16, 8))             # a layer that maps 16 numbers to 8
print("(5, 16) @ (16, 8) ->", (tokens @ W).shape)
print("all pairwise dot products, (5, 16) @ (16, 5) ->", (tokens @ tokens.T).shape)
try:
    tokens @ tokens
except ValueError as e:
    print("(5, 16) @ (5, 16) -> error:", str(e)[:60], "...")

## 4. Low rank: big matrices with little information

A 200 × 200 matrix has 40,000 numbers. If its rows are all mixtures of a few patterns, it can be written as two thin matrices instead. The SVD finds the best such approximation for any rank.

In [ ]:
structured = rng.normal(size=(200, 5)) @ rng.normal(size=(5, 200)) + 0.1 * rng.normal(size=(200, 200))
random = rng.normal(size=(200, 200))

ranks = [1, 2, 5, 10, 20, 50, 100, 200]
for name, M in [("5 patterns + noise", structured), ("pure noise", random)]:
    errs = [relative_error(M, np.matmul(*low_rank_approx(M, r))) for r in ranks]
    plt.plot(ranks, errs, "o-", label=name)
    print(f"{name:>18}: error at rank 5 = {errs[2]:.3f}, at rank 50 = {errs[5]:.3f}")
plt.xscale("log"); plt.xlabel("rank kept"); plt.ylabel("relative error"); plt.legend(); plt.show()

A, B = low_rank_approx(structured, 5)
print(f"rank 5 stores {A.size + B.size:,} numbers instead of {structured.size:,}")

## 5. LoRA: fine-tune with a low-rank update

Instead of changing a big weight matrix `W` directly, LoRA freezes it and learns a small update `B @ A`. Only `A` and `B` are trained.

In [ ]:
d = 4096
full = d * d
for r in [4, 16, 64]:
    n = lora_params(d, d, r)
    print(f"rank {r:>2}: {n:>9,} trainable numbers vs {full:,} ({100 * n / full:.2f}%)")

# The mechanics, at toy size
W = rng.normal(size=(8, 8))                              # frozen
B, A = np.zeros((8, 2)), rng.normal(size=(2, 8))         # trainable; B starts at zero
x = rng.normal(size=8)
print("starts identical to the original model:", np.allclose((W + B @ A) @ x, W @ x))

## Try it

- Add your own word vectors in section 1. Can you make two words with a high dot product but a low cosine similarity?
- In section 2, find a matrix that flips the square left to right. What does a matrix with a row of zeros do?
- In section 4, raise the noise from 0.1 to 1.0. What happens to the error at rank 5?
- A LoRA update of rank `r` can only change `W` in `r` directions. Why might that be enough for fine-tuning?